# 🔮 DỰ ĐOÁN NGUY CƠ RỚT MÔN – DSS AHP + Decision Tree

Nhập `Test_Score`, `Attendance (%)`, `Study_Hours` → Hệ thống trả về:
- **p_fail**: Xác suất rớt từ Decision Tree
- **ahp_score**: Điểm rủi ro có trọng số AHP
- **final_score**: Điểm kết hợp = 0.7 × p_fail + 0.3 × ahp_score
- **Mức rủi ro**: Cao / Trung bình / Thấp
- **Gợi ý can thiệp**

**Yêu cầu:** Phải chạy `train_model.ipynb` trước để tạo file mô hình `decision_tree_model.pkl`

## 1. Tải mô hình

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import joblib
import pandas as pd
import numpy as np

from modules.ahp import calculate_weights
from modules.risk_engine import (normalize_to_risk, calculate_final_score,
                                 classify_risk_level, generate_warning_reason,
                                 suggest_intervention)
from modules.ml_model import get_decision_path_for_student

MODEL_PATH = "decision_tree_model.pkl"
model = joblib.load(MODEL_PATH)
features = ["Test_Score", "Attendance (%)", "Study_Hours"]

# AHP weights (giống file train)
ahp_matrix = np.array([[1, 3, 5], [1/3, 1, 3], [1/5, 1/3, 1]])
ahp_weights, _, _, cr = calculate_weights(ahp_matrix)

print(f"✅ Đã tải mô hình Decision Tree (nhị phân)")
print(f"   Độ sâu: {model.get_depth()}, Số lá: {model.get_n_leaves()}")
print(f"   AHP weights: {dict(zip(features, np.round(ahp_weights, 3)))}")
print(f"   CR = {cr:.4f}")

✅ Đã tải mô hình Decision Tree từ: decision_tree_model.pkl
   Độ sâu cây: 4, Số lá: 7


## 2. Hàm dự đoán DSS (AHP + Decision Tree)

In [ ]:
def predict_student_dss(test_score, attendance, study_hours, alpha=0.7):
    """
    Dự đoán nguy cơ rớt môn kết hợp AHP + Decision Tree.
    Trả về: p_fail, ahp_score, final_score, mức rủi ro, rule path, gợi ý can thiệp.
    """
    X = pd.DataFrame([[test_score, attendance, study_hours]], columns=features)

    # 1. Decision Tree → p_fail
    p_fail = model.predict_proba(X)[0][1]

    # 2. AHP → ahp_score
    risk_values = {
        "Test_Score": 1 - test_score / 10.0,
        "Attendance (%)": 1 - attendance / 100.0,
        "Study_Hours": 1 - min(study_hours, 8.0) / 8.0,
    }
    ahp_score = sum(ahp_weights[i] * risk_values[features[i]] for i in range(len(features)))
    ahp_score = np.clip(ahp_score, 0, 1)

    # 3. Final score = alpha * p_fail + (1-alpha) * ahp_score
    final_score = np.clip(alpha * p_fail + (1 - alpha) * ahp_score, 0, 1)

    # 4. Mức rủi ro
    risk_level = classify_risk_level(final_score)

    # 5. Rule path
    rules = get_decision_path_for_student(model, X, features)

    # 6. Top factors
    top_factors = sorted(risk_values.items(), key=lambda x: ahp_weights[features.index(x[0])] * x[1], reverse=True)
    top_factors_weighted = [(k, ahp_weights[features.index(k)] * v) for k, v in top_factors]

    # 7. Gợi ý can thiệp
    reason = generate_warning_reason(top_factors_weighted[:3])
    suggestion = suggest_intervention(risk_level, top_factors_weighted[:3])

    # In kết quả
    icon = {"Cao": "🔴", "Trung bình": "🟡", "Thấp": "🟢"}.get(risk_level, "⚪")

    print("=" * 60)
    print("  KẾT QUẢ DỰ ĐOÁN – DSS AHP + DECISION TREE")
    print("=" * 60)
    print(f"  Điểm kiểm tra (Test_Score):    {test_score}")
    print(f"  Tỷ lệ đi học (Attendance):     {attendance}%")
    print(f"  Giờ học/ngày (Study_Hours):     {study_hours} giờ")
    print("-" * 60)
    print(f"  📊 p_fail (DT):       {p_fail:.4f}")
    print(f"  ⚖️ ahp_score:         {ahp_score:.4f}")
    print(f"  📐 final_score:       {final_score:.4f}")
    print(f"  {icon} Mức rủi ro:        {risk_level}")
    print("-" * 60)
    print(f"  🌲 Rule Path:")
    for i, r in enumerate(rules, 1):
        print(f"     Bước {i}: {r}")
    print("-" * 60)
    if risk_level in ("Cao", "Trung bình"):
        print(f"  ❗ Lý do cảnh báo:")
        print(f"  {reason}")
        print(f"\n  💡 Gợi ý can thiệp:")
        print(f"  {suggestion}")
    else:
        print(f"  ✅ Sinh viên không có nguy cơ rớt môn.")
    print("=" * 60)

    return {
        "p_fail": p_fail, "ahp_score": ahp_score,
        "final_score": final_score, "risk_level": risk_level,
    }

print("✅ Hàm predict_student_dss() đã sẵn sàng!")

✅ Hàm predict_student() đã sẵn sàng!


## 3. Demo – Dự đoán các trường hợp mẫu (với AHP + DT)

In [ ]:
print("📋 DEMO – DỰ ĐOÁN DSS AHP + DECISION TREE\n")

test_cases = [
    ("Sinh viên giỏi",         8.0, 95.0, 6.0),
    ("Sinh viên khá",          6.5, 85.0, 4.0),
    ("Sinh viên trung bình",   5.0, 75.0, 3.0),
    ("Sinh viên yếu",          3.5, 65.0, 1.5),
    ("Điểm OK nhưng hay vắng", 6.0, 60.0, 4.0),
    ("Ít học + điểm thấp",     4.5, 80.0, 1.5),
    ("Cần cù bù siêng",        4.0, 90.0, 5.0),
    ("Vắng rất nhiều",          5.0, 40.0, 3.0),
]

results = []
for name, test, attend, hours in test_cases:
    X = pd.DataFrame([[test, attend, hours]], columns=features)
    p_fail = model.predict_proba(X)[0][1]

    risk_vals = {
        "Test_Score": 1 - test / 10.0,
        "Attendance (%)": 1 - attend / 100.0,
        "Study_Hours": 1 - min(hours, 8.0) / 8.0,
    }
    ahp_score = sum(ahp_weights[i] * risk_vals[features[i]] for i in range(len(features)))
    ahp_score = np.clip(ahp_score, 0, 1)
    final = np.clip(0.7 * p_fail + 0.3 * ahp_score, 0, 1)
    level = classify_risk_level(final)
    icon = {"Cao": "🔴", "Trung bình": "🟡", "Thấp": "🟢"}.get(level, "⚪")

    results.append({
        "Trường hợp": name,
        "Điểm": test, "Đi học (%)": attend, "Giờ học": hours,
        "p_fail": round(p_fail, 3),
        "ahp_score": round(float(ahp_score), 3),
        "final_score": round(float(final), 3),
        "Mức rủi ro": f"{icon} {level}",
    })

pd.DataFrame(results)

📋 DEMO - DỰ ĐOÁN MỘT SỐ TRƯỜNG HỢP MẪU



,Trường hợp,Điểm,Đi học (%),Giờ học/ngày,Kết quả,% Đậu,% Nguy cơ,% Rớt
0,Sinh viên giỏi,8.0,95.0,6.0,ĐẬU,100.0%,0.0%,0.0%
1,Sinh viên khá,6.5,85.0,4.0,ĐẬU,100.0%,0.0%,0.0%
2,Sinh viên trung bình,5.0,75.0,3.0,ĐẬU,100.0%,0.0%,0.0%
3,Sinh viên yếu,3.5,65.0,1.5,RỚT MÔN,0.0%,0.0%,100.0%
4,Điểm OK nhưng hay vắng,6.0,60.0,4.0,CÓ NGUY CƠ,0.0%,100.0%,0.0%
5,Ít học + điểm thấp,4.5,80.0,1.5,ĐẬU,100.0%,0.0%,0.0%
6,Cần cù bù siêng,4.0,90.0,5.0,ĐẬU,100.0%,0.0%,0.0%
7,Vắng rất nhiều,5.0,40.0,3.0,CÓ NGUY CƠ,0.0%,100.0%,0.0%


## 4. Nhập điểm để dự đoán (DSS AHP + Decision Tree)
Thay đổi giá trị `test_score`, `attendance`, `study_hours` bên dưới rồi chạy cell

In [ ]:
# ===== THAY ĐỔI GIÁ TRỊ Ở ĐÂY =====
test_score  = 5.0    # Điểm kiểm tra giữa kỳ (0-10)
attendance  = 80.0   # Tỷ lệ đi học (%) (0-100)
study_hours = 3.0    # Số giờ học trung bình/ngày
# ======================================

result = predict_student_dss(test_score, attendance, study_hours)

print("=" * 55)
print(f"  Điểm: {test_score}  |  Chuyên cần: {attendance}%  |  Giờ học: {study_hours}h")
print("=" * 55)
print(f"  P(Rớt)      : {result['p_fail']:.4f}")
print(f"  AHP Score    : {result['ahp_score']:.4f}")
print(f"  Final Score  : {result['final_score']:.4f}")
print(f"  Mức rủi ro   : {result['risk_level']}")
print("-" * 55)
print(f"  Rule path    : {result['rule_path']}")
print(f"  Lý do        : {result['warning_reason']}")
print(f"  Đề xuất      : {result['intervention']}")
print("=" * 55)

  KẾT QUẢ DỰ ĐOÁN
  Điểm kiểm tra (Test_Score):    5.0
  Tỷ lệ đi học (Attendance):     80.0%
  Giờ học/ngày (Study_Hours):     3.0 giờ
--------------------------------------------------
  🟢 DỰ ĐOÁN: ✅ ĐẬU
  📊 Xác suất Đậu:          100.0%
  📊 Xác suất Có nguy cơ:   0.0%
  📊 Xác suất Rớt:          0.0%
